# Session 03 — Draw a helpful camera guide

**Today’s new skill:** use drawings on a webcam frame to tell a person what to do.

The system looks for an orange object, then draws a target and gives one clear response: **READY**, **ADJUST**, or **LOST**.

## Choose your webcam route

- **In Colab:** use the browser webcam preview, then capture a real webcam frame.
- **On your own computer:** run the continuous live-webcam loop.

Both routes use the same alignment rule and the same drawing code.

In [ ]:
# Run this once if OpenCV is not installed.
%pip install opencv-python

# OpenCV reads images and draws the visual guide.
import cv2
import numpy as np

## 1. Choose the rule you will test

Start with the suggested value. Later, change **only** `READY_TOLERANCE` and observe how the decision changes.

In [ ]:
# Orange colour range. Leave these two lines alone for the first test.
ORANGE_LOW = (4, 120, 90)
ORANGE_HIGH = (24, 255, 255)

# Size of the blue target box in the middle of the frame.
TARGET_WIDTH = 360
TARGET_HEIGHT = 240

# ★ YOUR FIRST EDIT: how close must the object be to the target centre?
# Try 24 first. Then try 50. Which value gives the most useful feedback?
READY_TOLERANCE = 24

## 2. Find the centre of the orange object

This helper returns the middle of the largest orange shape. If it cannot see an orange object, it returns `None`.

In [ ]:
def find_orange_centre(frame):
    # HSV makes it easier to keep only orange pixels.
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    orange_mask = cv2.inRange(hsv, ORANGE_LOW, ORANGE_HIGH)

    # Find orange shapes and keep the largest one.
    contours, _ = cv2.findContours(
        orange_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return None

    biggest_shape = max(contours, key=cv2.contourArea)
    if cv2.contourArea(biggest_shape) < 700:
        return None

    # Calculate the centre point of that shape.
    moments = cv2.moments(biggest_shape)
    if moments['m00'] == 0:
        return None
    centre_x = int(moments['m10'] / moments['m00'])
    centre_y = int(moments['m01'] / moments['m00'])
    return (centre_x, centre_y)

## 3. Draw the guide and choose a status

This helper adds the blue target, the centre cross, the measured point, and the message shown to the user.

In [ ]:
def draw_alignment_guide(frame):
    # Make a copy so the original frame stays unchanged.
    output = frame.copy()
    frame_height, frame_width = output.shape[:2]

    # Put the target box in the middle of this frame.
    left = (frame_width - TARGET_WIDTH) // 2
    top = (frame_height - TARGET_HEIGHT) // 2
    right = left + TARGET_WIDTH
    bottom = top + TARGET_HEIGHT
    target_centre = (left + TARGET_WIDTH // 2, top + TARGET_HEIGHT // 2)

    blue = (21, 138, 194)
    orange = (246, 138, 69)

    # Draw the visible guide for the user.
    cv2.rectangle(output, (left, top), (right, bottom), blue, 3)
    cv2.drawMarker(output, target_centre, orange, cv2.MARKER_CROSS, 20, 2)

    # Start with LOST. This changes only if we find an orange object.
    object_centre = find_orange_centre(output)
    status = 'LOST — show orange object'
    status_colour = orange

    if object_centre is not None:
        object_x, object_y = object_centre

        # Measure how far the orange object is from the target centre.
        distance = ((object_x - target_centre[0]) ** 2 +
                    (object_y - target_centre[1]) ** 2) ** 0.5
        inside_box = left <= object_x <= right and top <= object_y <= bottom
        ready = inside_box and distance <= READY_TOLERANCE

        # Draw the point that the system measured.
        status_colour = blue if ready else orange
        cv2.circle(output, object_centre, 8, status_colour, -1)
        cv2.line(output, target_centre, object_centre, status_colour, 2)

        # Choose the next action for the user.
        status = 'READY — capture frame' if ready else 'ADJUST — centre orange object'

    # Put the decision and the editable rule on the frame.
    cv2.putText(output, status, (18, 35), cv2.FONT_HERSHEY_SIMPLEX, .65, status_colour, 2)
    cv2.putText(output, f'tolerance: {READY_TOLERANCE}px', (18, frame_height - 18),
                cv2.FONT_HERSHEY_SIMPLEX, .6, (255, 255, 255), 2)
    return output, status

## 4A. Colab route — capture from your webcam

Run the next cell. Allow browser camera access, use the live preview to position your orange object, then press **CAPTURE FRAME**. Run this cell again whenever you want to test a new webcam frame.

The JavaScript below only connects Colab to your browser camera. You do not need to edit it.

In [ ]:
# This adapter opens the browser webcam after the learner clicks START CAMERA.
# You do not need to edit this cell.
from base64 import b64decode
from IPython.display import Javascript, display
from google.colab.output import eval_js

def capture_webcam_frame():
    display(Javascript(r'''
      async function captureFrame() {
        const panel = document.createElement('div');
        const start = document.createElement('button');
        const capture = document.createElement('button');
        const message = document.createElement('p');
        const video = document.createElement('video');

        start.textContent = 'START CAMERA';
        capture.textContent = 'CAPTURE FRAME';
        capture.disabled = true;
        start.style.cssText = capture.style.cssText =
          'padding:10px 16px;font-weight:bold;margin:8px 8px 8px 0;';
        video.style.cssText =
          'display:block;width:420px;max-width:100%;border:2px solid #10151a;';
        message.textContent = 'Click START CAMERA, then choose Allow in the browser prompt.';

        panel.appendChild(message);
        panel.appendChild(start);
        panel.appendChild(capture);
        panel.appendChild(video);
        document.body.appendChild(panel);

        // The browser allows camera access only after the learner clicks this button.
        const stream = await new Promise((resolve, reject) => {
          start.onclick = async () => {
            try {
              const webcam = await navigator.mediaDevices.getUserMedia({video: true});
              video.srcObject = webcam;
              await video.play();
              start.disabled = true;
              capture.disabled = false;
              message.textContent = 'Camera is live. Position the orange object, then click CAPTURE FRAME.';
              resolve(webcam);
            } catch (error) {
              message.textContent = 'Camera permission was denied. Use your browser site settings to allow the camera, then run this cell again.';
              reject(error);
            }
          };
        });

        // Wait until the learner chooses the exact frame to analyse.
        await new Promise(resolve => capture.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getTracks().forEach(track => track.stop());
        panel.remove();
        return canvas.toDataURL('image/jpeg');
      }
    '''))

    try:
        image_data = eval_js('captureFrame()')
    except Exception:
        # The browser message above explains how to allow the camera.
        return None

    image_bytes = b64decode(image_data.split(',')[1])
    return cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)

# Start the webcam, position an orange object, then click CAPTURE FRAME.
webcam_frame = capture_webcam_frame()

if webcam_frame is not None:
    result, status = draw_alignment_guide(webcam_frame)
    from google.colab.patches import cv2_imshow
    cv2_imshow(result)
    print(status)
else:
    print('No frame was captured. Run this cell again after allowing camera access.')

## 4B. Local route — continuous live webcam

Run this only on your own computer, in a local Python or Jupyter environment. Press **q** in the camera window to stop.

In [ ]:
# Open the first webcam on this computer.
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print('No local webcam was found. In Colab, use the webcam-capture cell above.')
else:
    while True:
        # Read one new frame, draw the guide, and show it.
        ok, frame = camera.read()
        if not ok:
            break
        result, status = draw_alignment_guide(frame)
        cv2.imshow('Alignment guide — press q to exit', result)

        # Press q in the camera window when you are finished.
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Close the camera safely after the loop ends.
    camera.release()
    cv2.destroyAllWindows()

## 5. Record evidence

1. With the object **centred**, what status appears?
2. With it **off-centre**, what status appears?
3. When it is **out of view**, what status appears?
4. After changing `READY_TOLERANCE`, does the guide approve too early, too late, or at a useful point?

**Exit evidence:** write one rule you changed, one visible result, and one case where a user could still be confused.